In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import ConvexHull


In [ ]:
# Read the CSV  

data = pd.read_csv("raman_spectra_api_compounds.csv", header=None)

In [ ]:
#SEPARATE THE DATA

raman_shift = data.iloc[0, :-1].astype(float).to_numpy()

spectra = data.iloc[1:, :-1]

labels = data.iloc[1:, -1]



In [ ]:
# In this plot we will Define the Baseline Correction 

def baseline_correction(x, y):

    order = np.argsort(x)

    x_sorted = x[order]
    y_sorted = y[order]

    hull = []

    for i in range(len(x_sorted)):

        while len(hull) >= 2:

            p1 = hull[-2]
            p2 = hull[-1]
            p3 = i

            cross = (
                (x_sorted[p2] - x_sorted[p1]) *
                (y_sorted[p3] - y_sorted[p1])
                -
                (y_sorted[p2] - y_sorted[p1]) *
                (x_sorted[p3] - x_sorted[p1])
            )

            if cross <= 0:
                hull.pop()
            else:
                break

        hull.append(i)

    baseline_x = x_sorted[hull]
    baseline_y = y_sorted[hull]

    baseline = np.interp(
        x_sorted,
        baseline_x,
        baseline_y
    )

    corrected = y_sorted - baseline

    return corrected


#  PROCESS ALL SPECTRA

processed_spectra = []

for i in range(len(spectra)):

    spectrum = spectra.iloc[i].astype(float).to_numpy()

    corrected = baseline_correction(
        raman_shift,
        spectrum
    )

    processed_spectra.append(corrected)


# Convert to NumPy array
processed_spectra = np.array(processed_spectra)



In [ ]:

# PEAK DETECTION Really not use for the Process data 


def peak_detection(x, corrected):
    
    # Ignore noisy region above 3100 cm⁻¹
    MAX_RAMAN_SHIFT = 3100
    
    valid = x < MAX_RAMAN_SHIFT
    
    x_clean = x[valid]
    corrected_clean = corrected[valid]

    peaks, properties = find_peaks(
        corrected_clean,
        prominence=500,
        distance=10
    )

    peak_positions = x_clean[peaks]
    peak_intensities = corrected_clean[peaks]

    return peak_positions, peak_intensities

In [ ]:

# PROCESS ALL SPECTRA


processed_spectra = []

for i in range(len(spectra)):

    spectrum = spectra.iloc[i].astype(float).to_numpy()

    corrected = baseline_correction(
        raman_shift,
        spectrum
    )

    processed_spectra.append(corrected)


# Convert to NumPy array
processed_spectra = np.array(processed_spectra)

# 5. CREATE PROCESSED DATASET
processed_data = pd.DataFrame(
    processed_spectra,
    columns=raman_shift
)

processed_data["Compound"] = labels.values


In [ ]:
# CHECK RESULTS
print("Processing complete.")
print("Number of spectra:", len(processed_spectra))
print("Number of Raman points:", len(raman_shift))
print("Saved as: processed_raman_data.csv")

In [ ]:
# Save Data on Desktop
import os

output_file = os.path.expanduser("~/Desktop/processed_raman_data.csv")

processed_data.to_csv(output_file, index=False)

print("Saved here:")
print(output_file)

In [ ]:
#Process Data 
import matplotlib.pyplot as plt

# Choose a few spectra to check
rows_to_plot = [0, 100, 1380, 2000]

for row in rows_to_plot:

    plt.figure(figsize=(10, 5))

    plt.plot(
        raman_shift,
        processed_data.iloc[row, :-1].astype(float)
    )

    plt.xlabel("Raman Shift (cm⁻¹)")
    plt.ylabel("Intensity")

    plt.title(
        "Processed Spectrum - "
        + str(processed_data.iloc[row]["Compound"])
    )

    plt.show()